# Task 1A: Naive RAG Pipeline

Basic RAG pipeline: fixed chunking (1024 tokens, overlap 200), dense retrieval (cosine), GPT-4o-mini generation.

**PDFs:** ktj.pdf (KTZh annual report), matnp_2024_rus.pdf (Maten Petroleum annual report)

In [1]:
import sys
sys.path.insert(0, "..")

import json
from src.parsing import parse_all_pdfs
from src.pipeline import RAGPipeline
from src.config import DEFAULT_CONFIG

## 1. Parse PDFs
Uses LlamaParse with markdown output. Results are cached to `data/parsed/`.

In [2]:
parsed_texts = parse_all_pdfs(use_cache=True)
for name, text in parsed_texts.items():
    print(f"{name}: {len(text)} chars")

ktj.pdf: 70944 chars
matnp_2024_rus.pdf: 49356 chars


## 2. Chunk & Index
Fixed chunking with 1024 tokens, 200 overlap. Embeddings: `intfloat/multilingual-e5-large`.

In [3]:
# Naive RAG config: fixed chunking, dense retrieval only
naive_config = {
    **DEFAULT_CONFIG,
    "chunking_strategy": "fixed",
    "chunk_size": 1024,
    "chunk_overlap": 200,
    "alpha": 1.0,  # dense only
    "use_reranking": False,
    "use_query_rewriting": False,
    "collection_name": "naive_rag",
}

pipeline = RAGPipeline(naive_config)
n_chunks = pipeline.ingest(parsed_texts)
print(f"Indexed {n_chunks} chunks")

Indexed 64 chunks


## 3. Demo: Single Query

In [4]:
result = pipeline.query("Каков был доход от основной деятельности АО «НК «КТЖ» в 2024 году?")
print("Answer:", result["answer"])
print("\n--- Retrieved chunks ---")
for i, ctx in enumerate(result["contexts"], 1):
    print(f"\nChunk {i} ({len(ctx)} chars):")
    print(ctx[:300], "...")

Answer: Консолидированная выручка АО «НК «ҚТЖ» в 2024 году составила 2 163,9 млрд тенге.

--- Retrieved chunks ---

Chunk 1 (965 chars):
С более подробной информацией о реализации Стратегии развития АО «НК «ҚТЖ» до 2032 года Вы можете ознакомиться на страницах Отчета с условным обозначением.

# Цели и задачи Компании на 2025 год

В 2025 году Компания планирует:

- увеличить грузооборот до 273,8 млрд т-км (4,6% к 2024 году);






- у ...

Chunk 2 (1938 chars):
# Единственный акционер

# Совет директоров

# Правление

# Управление рисками и внутренний контроль

# Служба внутреннего аудита

# Корпоративный омбудсмен

# Управление конфликтом интересов

# КОНСОЛИДИРОВАННАЯ ФИНАНСОВАЯ ОТЧЕТНОСТЬ

# ДОПОЛНИТЕЛЬНАЯ ИНФОРМАЦИЯ

# Об Отчете

# Ключевые показатели  ...

Chunk 3 (1993 chars):
2 Утверждены решением Совета директоров АО «НК «ҚТЖ» от 25 апреля 2024 года №4

3 Согласно Стратегии развития АО «НК «ҚТЖ» до 2032 года

4 Утверждены решением Совета директоров АО «НК «ҚТЖ» от 23 апреля 2025

## 4. Test on Golden Dataset (sample)

In [5]:
with open("../data/golden_dataset.json", "r", encoding="utf-8") as f:
    golden = json.load(f)

# Test on first 10 questions
for item in golden[:10]:
    result = pipeline.query(item["question"])
    print(f"Q: {item['question']}")
    print(f"Expected: {item['ground_truth']}")
    print(f"Got: {result['answer']}")
    print("-" * 80)

Q: Каков был доход от основной деятельности АО «НК «КТЖ» в 2024 году?
Expected: Доход от основной деятельности составил 2 163,9 млрд тенге.
Got: Консолидированная выручка АО «НК «ҚТЖ» в 2024 году составила 2 163,9 млрд тенге.
--------------------------------------------------------------------------------
Q: На сколько вырос доход от основной деятельности КТЖ в 2024 году по сравнению с 2023 годом (в абсолютном значении)?
Expected: Доход вырос на 229,8 млрд тенге.
Got: Консолидированная выручка АО «НК «ҚТЖ» в 2024 году составила 2 163,9 млрд тенге, что на 11,9% больше, чем в 2023 году. Для расчета абсолютного значения роста выручки:

1. Выручка в 2023 году: 
   \( \text{Выручка 2024} = \text{Выручка 2023} \times (1 + 0,119) \)
   \( 2 163,9 = \text{Выручка 2023} \times 1,119 \)
   \( \text{Выручка 2023} = \frac{2 163,9}{1,119} \approx 1 937,5 \) млрд тенге.

2. Абсолютный рост:
   \( 2 163,9 - 1 937,5 \approx 226,4 \) млрд тенге.

Таким образом, доход от основной деятельности КТЖ в 2024

## 5. Observed Problems

Common issues with Naive RAG:
1. **Tables split across chunks** — fixed chunking breaks table rows, losing context
2. **Exact names/numbers missed** — dense retrieval struggles with specific values (e.g., "1 875,6 млрд тенге")
3. **No keyword matching** — cosine similarity may miss lexically important terms
4. **Irrelevant chunks retrieved** — without reranking, top-K may include semantically similar but wrong passages

These problems motivate the Advanced RAG pipeline in Task 1B.